In [26]:
import pennylane as qml
from pennylane import numpy as np
from scipy.linalg import eigh

In [27]:
num = 2
device = qml.device('default.qubit',num)

In [28]:
weightage = [0.7,1.0,-0.2]
observable = [qml.PauliX(0),qml.PauliZ(1),qml.PauliZ(0)@qml.PauliZ(1)]
hamil = qml.Hamiltonian(weightage,observable)

In [29]:
def ansatz(theta):
   qml.RX(theta[0],wires=0)
   qml.RY(theta[1],wires=1)
   qml.CNOT(wires=[0,1])
   qml.RY(theta[2],wires=0)
   qml.RX(theta[3],wires=1)
@qml.qnode(device)
def find_expectation(theta,hamiltonian = hamil):
    ansatz(theta)
    return qml.expval(hamiltonian)

In [30]:
theta_g = np.random.uniform(0,10,4,requires_grad = True)
optimiser = qml.GradientDescentOptimizer(stepsize=0.3)
steps = 40
for i in range(steps):
    theta_g,cost = optimiser.step_and_cost(find_expectation,theta_g)
    if i%5==0:
        print(f'current paramter is {theta_g} and energy is {cost}')
print('\n')
print(f'final paramter is {theta_g} and ground_state energy defined by our hamiltonian is {find_expectation(theta_g)}')

current paramter is [7.92325544 6.65763361 5.94379051 2.88578196] and energy is 0.6503972627869262
current paramter is [6.54337842 6.51507402 5.43181504 3.04360801] and energy is -1.0928967631926374
current paramter is [6.28987964 6.33050207 4.76500058 3.13137648] and energy is -1.6622700951544096
current paramter is [6.28300609 6.2909301  4.53184328 3.14009284] and energy is -1.7222660527161218
current paramter is [6.28312117 6.28437233 4.46263411 3.14134321] and energy is -1.7275239611404698
current paramter is [6.28317354 6.28336379 4.44241618 3.14155067] and energy is -1.727969637295298
current paramter is [6.2831833  6.283212   4.43651816 3.14158559] and energy is -1.7280074729664372
current paramter is [6.28318497 6.28318929 4.43479778 3.14159147] and energy is -1.7280106898429117


final paramter is [6.28318523 6.28318618 4.43435371 3.14159237] and ground_state energy defined by our hamiltonian is -1.728010963483113


In [31]:
basis_states = [qml.Identity(0),qml.PauliX(1),qml.PauliY(0),qml.PauliZ(1)]
dim = len(basis_states)
h_mat = np.zeros((dim,dim),dtype = complex)
s_mat = np.zeros((dim,dim),dtype = complex)

In [32]:
for i in range(dim):
    for j in range(dim):
        s_mat[i][j] = find_expectation(theta_g,qml.prod(basis_states[i],basis_states[j]))
        # mat = qml.prod(basis_states[i],hamil,basis_states[j])
        # h_mat[i][j] = find_expectation(theta_g,mat)
        h_val=0.0
        for c,p in zip(weightage,observable):
            inter_mat = qml.prod(basis_states[i],p,basis_states[j])
            inter = c*find_expectation(theta_g,inter_mat)
            h_val+=inter
        h_mat[i][j]=h_val

In [33]:
print(s_mat)
print('\n')
print(h_mat)

[[ 1.00000000e+00+0.00000000e+00j  8.69854542e-07+0.00000000e+00j
   7.07212067e-14+0.00000000e+00j -1.00000000e+00+0.00000000e+00j]
 [ 8.69854542e-07+0.00000000e+00j  1.00000000e+00+0.00000000e+00j
   8.14703960e-08+0.00000000e+00j  1.38777838e-17+2.85234714e-07j]
 [ 7.07212067e-14+0.00000000e+00j  8.14703959e-08+0.00000000e+00j
   1.00000000e+00+0.00000000e+00j  0.00000000e+00+0.00000000e+00j]
 [-1.00000000e+00+0.00000000e+00j  1.38777918e-17-2.85234714e-07j
   0.00000000e+00+0.00000000e+00j  1.00000000e+00+0.00000000e+00j]]


[[-1.72801096e+00+0.00000000e+00j -5.85514387e-07-2.85223881e-07j
   1.05577778e-16+1.92477907e-04j  1.72801096e+00+0.00000000e+00j]
 [-5.85514387e-07+2.85223881e-07j  3.81775797e-01+0.00000000e+00j
   5.93283286e-08-1.67122397e-07j  9.17603798e-07-2.07649150e-07j]
 [ 1.10915711e-16-1.92477907e-04j  5.93283286e-08+1.67122397e-07j
  -2.71989037e-01+0.00000000e+00j  7.08822956e-14+1.92477907e-04j]
 [ 1.72801096e+00+0.00000000e+00j  9.17603798e-07+2.07649150e-07j


In [34]:
eigenvalues,eigenvectors = eigh(h_mat,s_mat)
print(eigenvalues)
print(eigenvectors)

[-1.72801099 -0.27198901  0.27324212  1.78545518]
[[-5.03850805e-01+7.14004306e-04j -1.74764917e+00+5.34004703e+00j
   1.11200833e+06-3.14790260e+06j  1.88628871e+04+1.20062621e+07j]
 [ 3.55318003e-09+4.28957587e-10j  3.04334929e-06-4.10568447e-06j
  -9.56745289e-01+2.74194074e+00j  3.67609633e+00-1.04495029e+01j]
 [ 1.78234573e-14-1.32194370e-04j  1.38656203e-10-9.99999991e-01j
   3.92672352e-08-1.08475300e-07j -1.54110006e-07+4.14826962e-07j]
 [ 4.96149186e-01+7.14004306e-04j -1.74778136e+00+5.34004703e+00j
   1.11200833e+06-3.14790260e+06j  1.88628871e+04+1.20062621e+07j]]
